In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# **Check DWH connection**

In [2]:
load_dotenv(find_dotenv())

PROJECT_ROOT = os.getenv("PROJECT_ROOT")
IPAI_PROJECT_DIR = os.getenv("IPAI_PROJECT_DIR") # <-- Читаем новую директорию

if not IPAI_PROJECT_DIR:
    raise ValueError("IPAI_PROJECT_DIR is not set in the .env file!")

if IPAI_PROJECT_DIR not in sys.path:
    sys.path.append(IPAI_PROJECT_DIR)

from src.database.connection_manager import db_manager

print("Testing connection to AWS RDS...")

try:
    with db_manager.get_dwh_connection() as conn:
        with conn.cursor() as cursor:
            
            cursor.execute("SELECT VERSION();")
            version = cursor.fetchone()
            
            print(f"Success! Connected to AWS RDS.")
            print(f"Database version: {version[0]}")
            
            cursor.execute("SHOW DATABASES;")
            databases = [db[0] for db in cursor.fetchall()]
            print(f"Available databases: {databases}")
            
except Exception as e:
    print(f"Test error: {e}")

Testing connection to AWS RDS...
Success! Connected to AWS RDS.
Database version: 8.4.8
Available databases: ['eliqsir_dwh', 'information_schema', 'mysql', 'performance_schema', 'sys']


# **DWH schema creation**

In [3]:
# 1. Load environment and set up paths
load_dotenv(find_dotenv())

# Fetch the sub-project directory where 'src' is located
IPAI_PROJECT_DIR = os.getenv("IPAI_PROJECT_DIR")

if not IPAI_PROJECT_DIR:
    raise ValueError("IPAI_PROJECT_DIR is not set in the .env file!")

# Add the sub-project directory to sys.path for imports
if IPAI_PROJECT_DIR not in sys.path:
    sys.path.append(IPAI_PROJECT_DIR)

from src.database.connection_manager import db_manager

# 2. Define the path to the schema.sql file using the new project directory
schema_file_path = os.path.join(IPAI_PROJECT_DIR, "src", "database", "schema.sql")

print(f"Looking for schema file at: {schema_file_path}")

try:
    # 3. Read the SQL file content
    with open(schema_file_path, 'r', encoding='utf-8') as file:
        sql_script = file.read()
    
    print("File loaded. Executing SQL statements on AWS RDS...")
    
    # 4. Split the script into individual statements
    # This bypasses the connector's 'multi' limitation
    sql_statements = sql_script.split(';')
    
    with db_manager.get_dwh_connection() as conn:
        # Use context manager for the cursor to ensure it closes properly
        with conn.cursor() as cursor:
            
            for statement in sql_statements:
                clean_statement = statement.strip()
                # Only execute if the statement isn't just empty space or newlines
                if clean_statement:
                    cursor.execute(clean_statement)
            
            # Commit the changes to the database
            conn.commit()
            
            print("Schema executed successfully!")
            
            # 5. Verification: Check if the tables were actually created
            cursor.execute("SHOW TABLES;")
            tables = [table[0] for table in cursor.fetchall()]
            
            print(f"\nTables currently in the 'eliqsir_dwh' database:")
            for table in tables:
                print(f" - {table}")
            
except FileNotFoundError:
    print(f"Error: Could not find the file 'schema.sql'. Check the path.")
except Exception as e:
    print(f"Execution error: {e}")

Looking for schema file at: C:\MariaSamosudova\Projects\UNIVER\repos\ELIQSIR\ipai_project\src\database\schema.sql
File loaded. Executing SQL statements on AWS RDS...
Schema executed successfully!

Tables currently in the 'eliqsir_dwh' database:
 - dim_article
 - dim_drug
 - dim_protein
 - dim_structure
 - fact_bioactivity


# **ETL**

## **Extraction**

**To run the extraction correctly you need to perform following steps to have chembl db locally:**
1. Go to the official European Bioinformatics Institute FTP site: ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/

2. Download the file named chembl_36_sqlite.tar.gz 
3. Unzip/extract that file locally to data/ChEMBL project folder

In [ ]:
# 1. Load environment variables
load_dotenv(find_dotenv())

# Fetch variables from the environment
PROJECT_ROOT = os.getenv("PROJECT_ROOT")
IPAI_PROJECT_DIR = os.getenv("IPAI_PROJECT_DIR")
DATA_DIR = os.getenv("DATA_DIR")
NCBI_EMAIL = os.getenv("NCBI_EMAIL")

# Ensure all critical variables are present
if not all([PROJECT_ROOT, IPAI_PROJECT_DIR, DATA_DIR, NCBI_EMAIL]):
    raise ValueError("Missing critical environment variables in the .env file!")

# Add the sub-project directory to sys.path so Python can find the 'src' module
if IPAI_PROJECT_DIR not in sys.path:
    sys.path.append(IPAI_PROJECT_DIR)

# 2. Set up the output directory
csv_dir = Path(DATA_DIR) / "csv"
csv_dir.mkdir(parents=True, exist_ok=True)

# Import the extractors from the src package
from src.etl.extraction import UniProtExtractor, ChemblExtractor, PdbeExtractor, PubMedExtractor

print(f"Starting ELIQSIR Extraction Pipeline...")
print(f"Output directory: {csv_dir}")

# -------------------------------------------------------------------
# STAGE 1: UniProt (The Foundation)
# -------------------------------------------------------------------
print("\n--- STAGE 1: Extracting UniProt Data ---")
uniprot_ext = UniProtExtractor()
df_uniprot = uniprot_ext.extract()

# Save raw CSV
df_uniprot.to_csv(csv_dir / "uniprot_raw.csv", index=False)

# Get the list of UniProt IDs for the next steps
uniprot_ids = df_uniprot['uniprot_id'].dropna().unique().tolist()

# TEST MODE: Keep only the first 50 proteins for a fast test run. 
# Remove or comment out the next line when you are ready to extract everything!
# uniprot_ids = uniprot_ids[:50] 
print(f"Proceeding with {len(uniprot_ids)} proteins for downstream extraction...")

# -------------------------------------------------------------------
# STAGE 2: ChEMBL (The Bioactivity & Drugs)
# -------------------------------------------------------------------
print("\n--- STAGE 2: Extracting ChEMBL Data ---")
# The ChEMBL extractor expects to find the 'chembl_36' folder inside this directory
chembl_base_dir = Path(DATA_DIR) / "ChEMBL" 

try:
    chembl_ext = ChemblExtractor(chembl_dir=chembl_base_dir)
    df_chembl = chembl_ext.extract(uniprot_ids=uniprot_ids)
    df_chembl.to_csv(csv_dir / "chembl_raw.csv", index=False)
    
    # Get PubMed IDs for Stage 3
    pubmed_ids = df_chembl['pubmed_id'].dropna().unique().tolist()
except FileNotFoundError as e:
    print(f"\nChEMBL Extraction Skipped: {e}")
    print("Please ensure your ChEMBL SQLite database is downloaded and extracted.")
    pubmed_ids = [] # Fallback if ChEMBL isn't downloaded yet

# -------------------------------------------------------------------
# STAGE 3: PubMed (The Articles)
# -------------------------------------------------------------------
print("\n--- STAGE 3: Extracting PubMed Data ---")
if pubmed_ids:
    pubmed_ext = PubMedExtractor(email=NCBI_EMAIL)
    df_pubmed = pubmed_ext.extract(pubmed_ids=pubmed_ids)
    df_pubmed.to_csv(csv_dir / "pubmed_raw.csv", index=False)
else:
    print("No PubMed IDs found (ChEMBL likely skipped). Skipping PubMed.")

# -------------------------------------------------------------------
# STAGE 4: PDBe (The 3D Structures)
# -------------------------------------------------------------------
print("\n--- STAGE 4: Extracting PDBe Data ---")
pdbe_ext = PdbeExtractor()
df_pdbe = pdbe_ext.extract(uniprot_ids=uniprot_ids)
df_pdbe.to_csv(csv_dir / "pdbe_raw.csv", index=False)

print(f"\nPipeline complete! All files saved to: {csv_dir}")

Starting ELIQSIR Extraction Pipeline...
Output directory: C:\MariaSamosudova\Projects\UNIVER\repos\ELIQSIR\data\csv

--- STAGE 1: Extracting UniProt Data ---
2026-03-29 21:47:55  INFO      src.extraction.uniprot_extractor  Fetching UniProt data – organism_id=9606, reviewed=True
2026-03-29 21:48:07  INFO      src.extraction.uniprot_extractor  UniProt extraction complete – 20431 proteins retrieved.
Proceeding with 20431 proteins for downstream extraction...

--- STAGE 2: Extracting ChEMBL Data ---
2026-03-29 21:48:08  INFO      src.extraction.chembl_extractor  ============================================================
2026-03-29 21:48:08  INFO      src.extraction.chembl_extractor  ChEMBL SQLite Extractor initialized
2026-03-29 21:48:08  INFO      src.extraction.chembl_extractor    Version  : 36
2026-03-29 21:48:08  INFO      src.extraction.chembl_extractor    Database : data\ChEMBL\chembl_36\chembl_36_sqlite\chembl_36.db
2026-03-29 21:48:08  INFO      src.extraction.chembl_extractor   